In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import glob
from astropy.io import fits
from matplotlib.widgets import Slider
from astropy.table import Table
import astropy.units as u
from astropy import constants as const

from reproject import reproject_interp
from scipy.ndimage import fourier_shift
from skimage.registration import phase_cross_correlation
from Functions import *
from astropy.io import fits
from astropy.wcs import WCS
from reproject import reproject_interp as rpj
from astropy.convolution import convolve, convolve_fft
from scipy.ndimage import zoom, shift as ndi_shift
from photutils.centroids import centroid_quadratic
import time
from photutils.aperture import CircularAperture, CircularAnnulus, aperture_photometry
from astropy.wcs.utils import proj_plane_pixel_area

%matplotlib widget

from ImageScience import ImageScience

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from astropy.io import fits
from astropy.visualization import ZScaleInterval
from scipy.optimize import curve_fit


# ============================================================
# 2D Gaussian model (used by curve_fit)
# ============================================================

def _gaussian2d_flat(xy, amp, x0, y0, sigma_x, sigma_y, theta, bg):
    """Rotated 2D Gaussian, returns a flat array for curve_fit."""
    xp, yp = xy
    ct, st = np.cos(theta), np.sin(theta)
    xr = (xp - x0) * ct + (yp - y0) * st
    yr = -(xp - x0) * st + (yp - y0) * ct
    return (bg + amp * np.exp(
        -0.5 * (xr**2 / sigma_x**2 + yr**2 / sigma_y**2)
    )).ravel()


# ============================================================
# Coarse centroid  (integer pixels, no external dependencies)
#
# Used only to snap the cutout centre onto the source before
# handing off to the 2D Gaussian fit, which finds the true
# sub-pixel centre as part of the optimisation.
# ============================================================

def coarse_centroid(image, x0, y0, search_box=6,
                    threshold_sigma=2.0, max_iter=10):
    """
    Iterative intensity-weighted centroid on a small stamp.
    Returns integer (cx, cy) within a few pixels of the source peak.
    """
    ny, nx = image.shape
    cx, cy = int(round(x0)), int(round(y0))

    for _ in range(max_iter):
        h  = search_box // 2
        x1 = max(0, cx - h);  x2 = min(nx, cx + h + 1)
        y1 = max(0, cy - h);  y2 = min(ny, cy + h + 1)

        stamp = image[y1:y2, x1:x2].copy()
        bg    = np.median(stamp)
        stamp -= bg

        mad   = np.median(np.abs(stamp))
        stamp = np.where(stamp > threshold_sigma * 1.4826 * mad, stamp, 0.0)

        total = stamp.sum()
        if total <= 0:
            break

        yy, xx   = np.indices(stamp.shape)
        cx_new   = int(round(x1 + (xx * stamp).sum() / total))
        cy_new   = int(round(y1 + (yy * stamp).sum() / total))
        cx_new   = max(0, min(nx - 1, cx_new))
        cy_new   = max(0, min(ny - 1, cy_new))

        if cx_new == cx and cy_new == cy:
            break
        cx, cy = cx_new, cy_new

    return cx, cy


# ============================================================
# 2D Gaussian fit on a cutout
#
# The fit centre (x0, y0) is a free parameter, so sub-pixel
# position errors in the coarse centroid do not affect FWHM.
# Both FWHM_x and FWHM_y are returned; the profile plot shows
# the elliptical model overlaid on binned radial data.
# ============================================================

def fit_2d_gaussian(cutout, fit_radius=15):
    """
    Fit a rotated 2D Gaussian to pixels within fit_radius of the
    cutout centre.

    Returns
    -------
    result : dict with keys
        fwhm_x, fwhm_y  – along the Gaussian principal axes (px)
        fwhm_mean       – geometric-mean FWHM = 2.3548 * sqrt(sx*sy)
        fwhm_major      – larger of fwhm_x, fwhm_y
        fwhm_minor      – smaller
        theta           – position angle of major axis (radians)
        sub_x, sub_y    – sub-pixel centre offset from cutout centre
        popt            – full parameter vector [amp,x0,y0,sx,sy,theta,bg]
    """
    ny, nx = cutout.shape
    cy, cx = ny // 2, nx // 2

    y_idx, x_idx = np.indices(cutout.shape)
    r = np.sqrt((x_idx - cx)**2 + (y_idx - cy)**2)

    # Background from outer annulus
    rmax    = min(cx, cy)
    outer   = r >= max(rmax - 5, rmax * 0.8)
    bg_est  = float(np.median(cutout[outer])) if outer.any() else 0.0

    # Pixels used for fitting
    mask = r <= fit_radius
    xd   = x_idx[mask].ravel().astype(float)
    yd   = y_idx[mask].ravel().astype(float)
    zd   = cutout[mask].ravel().astype(float)

    amp_est = float(np.max(zd) - bg_est)
    if amp_est <= 0:
        raise ValueError("No positive signal in fit window")

    p0 = [amp_est, float(cx), float(cy), 2.0, 2.0, 0.0, bg_est]
    lo = [0,        cx - fit_radius, cy - fit_radius,
          0.3,  0.3,  -np.pi / 2, -np.inf]
    hi = [np.inf,   cx + fit_radius, cy + fit_radius,
          fit_radius, fit_radius, np.pi / 2,  np.inf]

    popt, _ = curve_fit(
        _gaussian2d_flat, (xd, yd), zd,
        p0=p0, bounds=(lo, hi), maxfev=20000,
    )

    amp, x0f, y0f, sx, sy, theta, bg = popt
    sx, sy = abs(sx), abs(sy)

    fwhm_x = 2.3548 * sx
    fwhm_y = 2.3548 * sy
    fwhm_mean  = 2.3548 * np.sqrt(sx * sy)
    fwhm_major = max(fwhm_x, fwhm_y)
    fwhm_minor = min(fwhm_x, fwhm_y)

    return dict(
        fwhm_x=fwhm_x, fwhm_y=fwhm_y,
        fwhm_mean=fwhm_mean,
        fwhm_major=fwhm_major, fwhm_minor=fwhm_minor,
        theta=theta,
        sub_x=x0f - cx, sub_y=y0f - cy,
        popt=popt,
    )


# ============================================================
# Helpers
# ============================================================

def percentile_stretch(image, lo=1, hi=99.5):
    data = np.nan_to_num(image, nan=0.0, posinf=0.0, neginf=0.0)
    vmin = np.percentile(data, lo)
    vmax = np.percentile(data, hi)
    return data, vmin, max(vmax, vmin + 1)


def _binned_radial_profile(cutout, cx_sub, cy_sub, fit_radius):
    """Binned radial profile centred on the sub-pixel fit centre."""
    ny, nx = cutout.shape
    y_idx, x_idx = np.indices(cutout.shape)
    r = np.sqrt((x_idx - cx_sub)**2 + (y_idx - cy_sub)**2)
    r_int = r.astype(int)
    rmax  = int(np.floor(fit_radius)) + 1
    profile = np.full(rmax, np.nan)
    for i in range(rmax):
        m = r_int == i
        if m.any():
            profile[i] = np.mean(cutout[m])
    valid = ~np.isnan(profile)
    return np.arange(rmax)[valid], profile[valid]


# ============================================================
# Core: centroid + fit + display
# ============================================================

def process_click(
    x0, y0,
    full_image,
    half, fit_radius,
    nx_full, ny_full,
    ax_cutout, ax_profile,
    ov_marker,           # matplotlib artist on overview panel
    zscale, fig, search_box=6
):
    """
    1. Coarse centroid to snap onto nearest source (integer px).
    2. Extract cutout centred on that integer position.
    3. Fit a 2D Gaussian with free sub-pixel centre.
    4. Update all three display panels.

    Returns (cutout, cx, cy) on success, None on failure.
    """
    # --- guard edges -----------------------------------------
    if (x0 < half or y0 < half
            or x0 > nx_full - half - 1
            or y0 > ny_full - half - 1):
        print("Click too close to image edge – ignored.")
        return None

    # --- coarse centroid -------------------------------------
    cx, cy = coarse_centroid(full_image, x0, y0,search_box=search_box)
    print(f"  Coarse centroid → ({cx}, {cy})")

    if (cx < half or cy < half
            or cx > nx_full - half - 1
            or cy > ny_full - half - 1):
        print("  Centroid too close to edge – ignored.")
        return None

    # --- cutout (centred on integer centroid) ----------------
    cutout = full_image[
        cy - half: cy + half + 1,
        cx - half: cx + half + 1,
    ].copy()

    # --- 2D Gaussian fit -------------------------------------
    try:
        res = fit_2d_gaussian(cutout, fit_radius=fit_radius)
        fit_ok = True
    except Exception as e:
        print(f"  2D fit failed: {e}")
        fit_ok = False
        res = {}

    # --- update overview marker ------------------------------
    if ov_marker is not None:
        ov_marker.set_data([cx], [cy])

    # --- cutout panel ----------------------------------------
    ax_cutout.clear()
    try:
        vc_min, vc_max = zscale.get_limits(cutout)
    except Exception:
        _, vc_min, vc_max = percentile_stretch(cutout)

    ax_cutout.imshow(
        cutout, origin='lower', cmap='inferno',
        vmin=vc_min, vmax=vc_max, interpolation='nearest',
    )

    if fit_ok:
        # Mark the sub-pixel fit centre
        fx = half + res['sub_x']
        fy = half + res['sub_y']
        ax_cutout.plot(fx, fy, '+', color='cyan', ms=5, mew=0.5)

        # Draw the FWHM ellipse
        from matplotlib.patches import Ellipse
        ellipse = Ellipse(
            xy=(fx, fy),
            width=res['fwhm_x'], height=res['fwhm_y'],
            angle=np.degrees(res['theta']),
            edgecolor='lime', facecolor='none', lw=0.5, alpha=0.5,
        )
        ax_cutout.add_patch(ellipse)
    else:
        ax_cutout.axhline(half, color='cyan', lw=0.4, alpha=0.5)
        ax_cutout.axvline(half, color='cyan', lw=0.4, alpha=0.5)

    ax_cutout.set_title(
        f"Cutout  (x={cx}, y={cy})\nClick here to refine", fontsize=9
    )

    # --- profile panel ---------------------------------------
    ax_profile.clear()

    if fit_ok:
        # Background-subtracted cutout for profile
        bg_val = res['popt'][6]
        cutout_bs = cutout - bg_val

        r_bins, prof = _binned_radial_profile(
            cutout_bs,
            half + res['sub_x'],
            half + res['sub_y'],
            fit_radius,
        )

        # Normalise
        amp = res['popt'][0]
        r_fine = np.linspace(0, fit_radius, 400)
        gauss1d = amp * np.exp(-0.5 * r_fine**2 /
                               (res['popt'][3] * res['popt'][4]))  # geometric sigma

        ax_profile.scatter(r_bins, prof / amp, s=18,
                           color='steelblue', zorder=3, label='Radial profile')
        ax_profile.plot(r_fine, gauss1d / amp, '-', lw=2, color='tomato',
                        label=(f"2D Gaussian fit\n"
                               f"FWHM_max = {res['fwhm_major']:.2f} px\n"
                               f"FWHM_min = {res['fwhm_minor']:.2f} px\n"
                               f"FWHM_mean = {res['fwhm_mean']:.2f} px"))
        ax_profile.axhline(0.5, color='gray', lw=0.8, ls='--', alpha=0.6)
        title = (f"FWHM  maj={res['fwhm_major']:.2f}"
                 f"min={res['fwhm_minor']:.2f}  "
                 f"mean={res['fwhm_mean']:.2f} px")
    else:
        title = "Fit failed"

    ax_profile.set_xlabel("Radius (pixels)", fontsize=9)
    ax_profile.set_ylabel("Normalised Intensity", fontsize=9)
    ax_profile.set_title(title, fontsize=9)
    if fit_ok:
        ax_profile.legend(fontsize=8)
    ax_profile.set_ylim(-0.15, 1.25)

    fig.canvas.draw_idle()

    eccentricity = 1-(res['fwhm_minor']/res['fwhm_major'])
    
    if fit_ok:
        print(f"  x={cx}, y={cy}  |  "
              f"FWHM_max={res['fwhm_major']:.3f}  "
              f"FWHM_min={res['fwhm_minor']:.3f}  "
              f"FWHM_mean={res['fwhm_mean']:.3f} px  "
              f"Eccentricity={eccentricity:.3f}   "
              f"sub-pixel offset=({res['sub_x']:+.2f}, {res['sub_y']:+.2f})")
    
    return cutout, cx, cy


# ============================================================
# Main inspector
# ============================================================

def interactive_fwhm_inspector(
    image_file,
    cutout_size=101,
    fit_radius=15,
    overview_size=1500,
):
    """
    Two-level interactive FWHM inspector.

    Left panel   – downsampled overview; click to navigate.
    Centre panel – full-resolution cutout with FWHM ellipse;
                   click to refine (re-centroid + re-fit).
    Right panel  – binned radial profile + 2D Gaussian fit.

    Parameters
    ----------
    image_file   : path to FITS file
    cutout_size  : side length of analysis cutout in pixels (odd)
    fit_radius   : radius in pixels used for the 2D Gaussian fit
    overview_size: longest axis of the downsampled overview (px)
    """
    print("Loading FITS …")
    full_image = fits.getdata(image_file).astype(float)
    full_image = np.nan_to_num(full_image, nan=0.0, posinf=0.0, neginf=0.0)
    ny_full, nx_full = full_image.shape

    factor   = max(1, max(ny_full, nx_full) // overview_size)
    overview = full_image[::factor, ::factor]

    print(f"Full image : {nx_full}×{ny_full}")
    print(f"Overview   : {overview.shape[1]}×{overview.shape[0]}  (1/{factor})")

    half = cutout_size // 2

    zscale = ZScaleInterval(n_samples=10000, contrast=0.25)
    try:
        vmin_ov, vmax_ov = zscale.get_limits(overview)
    except Exception:
        _, vmin_ov, vmax_ov = percentile_stretch(overview)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    ax_image, ax_cutout, ax_profile = axes

    ax_image.imshow(
        overview, origin='lower', cmap='gray',
        vmin=vmin_ov, vmax=vmax_ov, interpolation='nearest',
    )
    ax_image.set_title("Overview — click to select source\n(ZScale stretch)", fontsize=9)

    # Overview marker shown in full-image pixel coords
    ov_marker, = ax_image.plot([], [], 'r+', ms=14, mew=1.5,
                               transform=ax_image.transData)

    # The overview axes are in downsampled coords, so we need a wrapper
    # that converts the marker position from full-image → overview coords.
    class _OvMarker:
        def set_data(self, xs, ys):
            ov_marker.set_data(
                [x / factor for x in xs],
                [y / factor for y in ys],
            )

    ov_marker_wrapped = _OvMarker()

    ax_cutout.set_title("Cutout", fontsize=9)
    ax_profile.set_title("Radial Profile", fontsize=9)

    state = {'cutout': None, 'x0': None, 'y0': None}

    # --------------------------------------------------------
    # Overview click → navigate
    # --------------------------------------------------------
    def onclick_overview(event):
        if event.inaxes != ax_image:
            return
        x0 = int(round(event.xdata)) * factor
        y0 = int(round(event.ydata)) * factor
        result = process_click(
            x0, y0, full_image, half, fit_radius, nx_full, ny_full,
            ax_cutout, ax_profile, ov_marker_wrapped, zscale, fig,
        )
        if result is not None:
            state['cutout'], state['x0'], state['y0'] = result
            fig.canvas.draw_idle()

    # --------------------------------------------------------
    # Cutout click → refine
    # --------------------------------------------------------
    def onclick_cutout(event):
        if event.inaxes != ax_cutout or state['cutout'] is None:
            return
        dx = int(round(event.xdata)) - half
        dy = int(round(event.ydata)) - half
        x0_new = state['x0'] + dx
        y0_new = state['y0'] + dy
        result = process_click(
            x0_new, y0_new, full_image, half, fit_radius, nx_full, ny_full,
            ax_cutout, ax_profile, ov_marker_wrapped, zscale, fig,
        )
        if result is not None:
            state['cutout'], state['x0'], state['y0'] = result
            fig.canvas.draw_idle()

    fig.canvas.mpl_connect('button_press_event', onclick_overview)
    fig.canvas.mpl_connect('button_press_event', onclick_cutout)

    plt.tight_layout()
    plt.show()


#jwst_file = '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_i2d_anchor.fits'
#hst_file = '/project/galaxies/tjuchau/data_files/HST/ngc5194/F658N_HST_ACS_WFC_IVM_drc.fits'


jwst_file = '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc1433/hlsp_phangs-hst_hst_wfc3-uvis_ngc1433_f555w_v1_exp-drc-sci.fits'
hst_file = '/project/galaxies/tjuchau/data_files/HST/ngc1433/hlsp_phangs-hst_hst_wfc3-uvis_ngc1433_f555w_v1_exp-drc-sci.fits'


#jwst_file = '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc1512/hlsp_phangs-hst_hst_wfc3-uvis_ngc1512mosaic_f555w_v1_exp-drc-sci.fits'
#hst_file = '/project/galaxies/tjuchau/data_files/HST/ngc1512/hlsp_phangs-hst_hst_wfc3-uvis_ngc1512mosaic_f555w_v1_exp-drc-sci.fits'


#jwst_file = '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc1672/ngc5194_nircam_lv3_f187n_i2d_anchor.fits'
#hst_file = '/project/galaxies/tjuchau/data_files/HST/ngc1672/hlsp_phangs-hst_hst_wfc3-uvis_ngc1672mosaic_f555w_v1_exp-drc-sci.fits'


interactive_fwhm_inspector(
    ngc1672.files['pa_cont_aligned'],
    cutout_size=500,
    fit_radius=15,
    overview_size=1500,
)

In [ ]:
#TJ inspect continuum subtractions

redo=False #TJ takes about 45mins
m51_hst_scales = [1.221]
ngc1672_hst_scales = [1.343, 1.271, ]
ngc1672_jwst_scales = []
m51_hst_potential_stars = [(9838, 4106), (9894, 4082)]
m51_jwst_potential_stars = [(8559, 2335)]
ngc1433_hststars = [(6206, 6158), (4617, 5129), (6148, 4986)]
ngc_1672_jwststars = [(5015, 3625), (5256, 3670), (6177, 4569), (6130, 5773)]
stars = [(5015, 3625), (5256, 3670), (6177, 4569), (6130, 5773)]
if redo:
    #TJ m51 object
    jwst_cont = '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f150w_i2d_anchor.fits'
    jwst_feature = '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_i2d_anchor.fits'
    kernel_path = '/project/galaxies/tjuchau/data_files/JWST/PSFs/F150W_to_F187N.fits'
    m51 = ImageScience()
    m51.load_image('f150', jwst_cont)
    m51.load_image('f187', jwst_feature)
    m51.align_images('f187', 'f150') #TJ creates name of second file_aligned
    m51.fft_convolve('f150_aligned', kernel_path, return_time=True, out_name='f150_convolved')
    hst_cont = '/project/galaxies/tjuchau/data_files/HST/ngc5194/F555W_HST_ACS_WFC_IVM_drc.fits'
    hst_feature = '/project/galaxies/tjuchau/data_files/HST/ngc5194/F658N_HST_ACS_WFC_IVM_drc.fits'
    m51.load_image('f555', hst_cont)
    m51.load_image('f658', hst_feature)
    m51.align_images('f658', 'f555') #TJ creates name of second file_aligned

    #TJ ngc1433 object
    jwst_cont = '/project/galaxies/tjuchau/data_files/JWST/images/ngc1433/ngc1433_nircam_lv3_f150w_i2d_anchor.fits'
    jwst_feature = '/project/galaxies/tjuchau/data_files/JWST/images/ngc1433/ngc1433_nircam_lv3_f187n_i2d_anchor.fits'
    kernel_path = '/project/galaxies/tjuchau/data_files/JWST/PSFs/F150W_to_F187N.fits'
    ngc1433 = ImageScience()
    ngc1433.load_image('f150', jwst_cont)
    ngc1433.load_image('f187', jwst_feature)
    ngc1433.align_images('f187', 'f150') #TJ creates name of second file_aligned
    ngc1433.fft_convolve('f150_aligned', kernel_path, return_time=True, out_name='f150_convolved')
    hst_cont = '/project/galaxies/tjuchau/data_files/HST/ngc1433/hlsp_phangs-hst_hst_wfc3-uvis_ngc1433_f555w_v1_exp-drc-sci.fits'
    hst_feature = '/project/galaxies/tjuchau/data_files/HST/ngc1433/ngc1433_uvis_f658n_ivm_drc_sci.fits'
    ngc1433.load_image('f555', hst_cont)
    ngc1433.load_image('f658', hst_feature)
    ngc1433.align_images('f658', 'f555') #TJ creates name of second file_aligned


    #TJ ngc1512 object
    jwst_cont = '/project/galaxies/tjuchau/data_files/JWST/images/ngc1512/ngc1512_nircam_lv3_f150w_i2d_anchor.fits'
    jwst_feature = '/project/galaxies/tjuchau/data_files/JWST/images/ngc1512/ngc1512_nircam_lv3_f187n_i2d_anchor.fits'
    kernel_path = '/project/galaxies/tjuchau/data_files/JWST/PSFs/F150W_to_F187N.fits'
    ngc1512 = ImageScience()
    ngc1512.load_image('f150', jwst_cont)
    ngc1512.load_image('f187', jwst_feature)
    ngc1512.align_images('f187', 'f150') #TJ creates name of second file_aligned
    ngc1512.fft_convolve('f150_aligned', kernel_path, return_time=True, out_name='f150_convolved')
    hst_cont = '/project/galaxies/tjuchau/data_files/HST/ngc1512/hlsp_phangs-hst_hst_wfc3-uvis_ngc1512mosaic_f555w_v1_exp-drc-sci.fits'
    hst_feature = '/project/galaxies/tjuchau/data_files/HST/ngc1512/ngc1512_uvis_f658n_ivm_drc_sci.fits'
    ngc1512.load_image('f555', hst_cont)
    ngc1512.load_image('f658', hst_feature)
    ngc1512.align_images('f658', 'f555') #TJ creates name of second file_aligned

    #TJ ngc1672 object
    ngc1672 = ImageScience()
    ngc1672.load_image('f187', '/project/galaxies/tjuchau/data_files/JWST/images/ngc1672/ngc1672_nircam_lv3_f187n_i2d_anchor.fits')
    ngc1672.load_image('f150', '/project/galaxies/tjuchau/data_files/JWST/images/ngc1672/ngc1672_nircam_lv3_f150w_i2d_anchor_aligned_convolved.fits')
    ngc1672.load_image('f555', '/project/galaxies/tjuchau/data_files/HST/ngc1672/F555W_HST_ACS_WFC_IVM_drc.fits')
    ngc1672.load_image('f658', '/project/galaxies/tjuchau/data_files/HST/ngc1672/F658N_HST_ACS_WFC_IVM_drc.fits')
    ngc1672.save_fits('f555', '/project/galaxies/tjuchau/data_files/HST/ngc1672/F555W_HST_ACS_WFC_IVM_drc.fits', scale = 1.275)
    ngc1672.load_image('f555_scaled', '/project/galaxies/tjuchau/data_files/HST/ngc1672/F555W_HST_ACS_WFC_IVM_drc.fits')
    ngc1672.load_image('pa_cont', '/project/galaxies/tjuchau/data_files/JWST/PHANGS_products/hlsp_4793_jwst_nircam_ngc1672_f187n_v4p1_line-atf150wxf187nxf200w.fits', hdu='CONTINUUM')
    ngc1672.load_image('pa_contsub', '/project/galaxies/tjuchau/data_files/JWST/PHANGS_products/hlsp_4793_jwst_nircam_ngc1672_f187n_v4p1_line-atf150wxf187nxf200w.fits', hdu='CONTSUB')
    jwst_cont = '/project/galaxies/tjuchau/data_files/JWST/images/ngc1672/ngc1672_nircam_lv3_f150w_i2d_anchor.fits'
    ngc1672.align_images('f187', 'pa_cont')
    ngc1672.align_images('f658', 'f555')
    print(f'Loaded galaxy ngc1672 with callable keys :\n{list(ngc1672.images.keys())}')

else:
    try:
        test = [m51.images['f150_convolved'], m51.images['f187'], m51.images['f555'], m51.images['f658']]
        test = [ngc1433.images['f150_convolved'], ngc1433.images['f187'], ngc1433.images['f555'], ngc1433.images['f658']]
        test = [ngc1512.images['f150_convolved'], ngc1512.images['f187'], ngc1512.images['f555'], ngc1512.images['f658']]
        test = [ngc1672.images['f150_convolved'], ngc1672.images['f187'], ngc1672.images['f555'], ngc1672.images['f658']]
    except:
        m51 = ImageScience()
        m51.load_image('f555', '/project/galaxies/tjuchau/data_files/HST/ngc5194/F555W_HST_ACS_WFC_IVM_drc.fits')
        m51.load_image('f658', '/project/galaxies/tjuchau/data_files/HST/ngc5194/F658N_HST_ACS_WFC_IVM_drc.fits')
        m51.load_image('f150_convolved', '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f150w_i2d_anchor_aligned_convolved.fits')
        m51.load_image('f187', '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_i2d_anchor.fits')
        
        ngc1433 = ImageScience()
        ngc1433.load_image('f555', '/project/galaxies/tjuchau/data_files/HST/ngc1433/hlsp_phangs-hst_hst_wfc3-uvis_ngc1433_f555w_v1_exp-drc-sci.fits')
        ngc1433.load_image('f658', '/project/galaxies/tjuchau/data_files/HST/ngc1433/ngc1433_uvis_f658n_ivm_drc_sci.fits')
        ngc1433.load_image('f150_convolved', '/project/galaxies/tjuchau/data_files/JWST/images/ngc1433/ngc1433_nircam_lv3_f150w_i2d_anchor_aligned_convolved.fits')
        ngc1433.load_image('f187', '/project/galaxies/tjuchau/data_files/JWST/images/ngc1433/ngc1433_nircam_lv3_f187n_i2d_anchor.fits')

        ngc1512 = ImageScience()
        ngc1512.load_image('f555', '/project/galaxies/tjuchau/data_files/HST/ngc1512/hlsp_phangs-hst_hst_wfc3-uvis_ngc1512mosaic_f555w_v1_exp-drc-sci.fits')
        ngc1512.load_image('f658', '/project/galaxies/tjuchau/data_files/HST/ngc1512/ngc1512_uvis_f658n_ivm_drc_sci.fits')
        ngc1512.load_image('f150_convolved', '/project/galaxies/tjuchau/data_files/JWST/images/ngc1512/ngc1512_nircam_lv3_f150w_i2d_anchor_aligned_convolved.fits')
        ngc1512.load_image('f187', '/project/galaxies/tjuchau/data_files/JWST/images/ngc1512/ngc1512_nircam_lv3_f187n_i2d_anchor.fits')

        ngc1672 = ImageScience()
        ngc1672.load_image('f187', '/project/galaxies/tjuchau/data_files/JWST/images/ngc1672/ngc1672_nircam_lv3_f187n_i2d_anchor.fits')
        ngc1672.load_image('f150', '/project/galaxies/tjuchau/data_files/JWST/images/ngc1672/ngc1672_nircam_lv3_f150w_i2d_anchor_aligned_convolved.fits')
        ngc1672.load_image('f555', '/project/galaxies/tjuchau/data_files/HST/ngc1672/F555W_HST_ACS_WFC_IVM_drc.fits')
        ngc1672.load_image('f658', '/project/galaxies/tjuchau/data_files/HST/ngc1672/F658N_HST_ACS_WFC_IVM_drc.fits')
        ngc1672.save_fits('f555', '/project/galaxies/tjuchau/data_files/HST/ngc1672/F555W_HST_ACS_WFC_IVM_drc.fits', scale = 1.275)
        ngc1672.load_image('f555_scaled', '/project/galaxies/tjuchau/data_files/HST/ngc1672/F555W_HST_ACS_WFC_IVM_drc.fits')
        ngc1672.load_image('pa_cont', '/project/galaxies/tjuchau/data_files/JWST/PHANGS_products/hlsp_4793_jwst_nircam_ngc1672_f187n_v4p1_line-atf150wxf187nxf200w.fits', hdu='CONTINUUM')
        ngc1672.load_image('pa_contsub', '/project/galaxies/tjuchau/data_files/JWST/PHANGS_products/hlsp_4793_jwst_nircam_ngc1672_f187n_v4p1_line-atf150wxf187nxf200w.fits', hdu='CONTSUB')
        jwst_cont = '/project/galaxies/tjuchau/data_files/JWST/images/ngc1672/ngc1672_nircam_lv3_f150w_i2d_anchor.fits'
        ngc1672.align_images('f187', 'pa_cont')
        ngc1672.align_images('f658', 'f555')
        print(f'Loaded galaxy ngc1672 with callable keys :\n{list(ngc1672.images.keys())}')
ngc1672.inspect_continuum_subtraction('cont', 'f187', initial_scale=1.221, zoom_size=100, zoom_center=ngc_1672_jwststars[-1])

In [ ]:
hdu = fits.open('/project/galaxies/tjuchau/data_files/JWST/PHANGS_products/hlsp_4793_jwst_nircam_ngc1672_f187n_v4p1_line-atf150wxf187nxf200w.fits')
data = hdu['CONTINUUM'].data
plt.imshow(data)

In [ ]:
table = Table.read('/project/galaxies/tjuchau/data_files/Kiana_Cluster_Files/full_table_ryan.csv')
table = table[table['galaxy']=="M51"]
table[0]

In [ ]:
m51 = ImageScience()
m51.load_image('f555', '/project/galaxies/tjuchau/data_files/HST/ngc5194/F555W_HST_ACS_WFC_IVM_drc.fits')
m51.load_image('f658', '/project/galaxies/tjuchau/data_files/HST/ngc5194/F658N_HST_ACS_WFC_IVM_drc.fits')
m51.load_image('f150_convolved', '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f150w_i2d_anchor_aligned_convolved.fits')
m51.load_image('f187', '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_i2d_anchor.fits')
m51.save_fits('f150_convolved', '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f150w_i2d_anchor_aligned_convolved_rescaled.fits', scale=1)
m51.save_fits('f555', '/project/galaxies/tjuchau/data_files/HST/ngc5194/F555W_HST_ACS_WFC_IVM_drc_rescaled.fits', scale=1.221)
m51.load_image('f150_scaled', '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f150w_i2d_anchor_aligned_convolved_rescaled.fits')
m51.load_image('f555_scaled', '/project/galaxies/tjuchau/data_files/HST/ngc5194/F555W_HST_ACS_WFC_IVM_drc_rescaled.fits')




In [ ]:
pa_EW = []
ha_EW = []
for row in table:
    pa_EW.append(m51.get_equivalent_width('f187', 'f150_scaled', [row['ra'], row['dec']], row['radius']*u.arcsec, 0.1*u.arcsec)[0].value)
    ha_EW.append(m51.get_equivalent_width('f658', 'f555', [row['ra'], row['dec']], row['radius']*u.arcsec, 0.1*u.arcsec)[0].value)
table.add_column(pa_EW, name='pa_EW')
table.add_column(ha_EW, name='ha_EW')
plt.clf()
plt.scatter(pa_EW, ha_EW)
plt.xlabel('pa_EW')
plt.ylabel('ha_EW')
plt.show()


In [ ]:
table.add_column?

In [ ]:
ngc1672 = ImageScience()
ngc1672.load_image('f187', '/project/galaxies/tjuchau/data_files/JWST/images/ngc1672/ngc1672_nircam_lv3_f187n_i2d_anchor.fits')
ngc1672.load_image('f150', '/project/galaxies/tjuchau/data_files/JWST/images/ngc1672/ngc1672_nircam_lv3_f150w_i2d_anchor_aligned_convolved.fits')
ngc1672.load_image('f555', '/project/galaxies/tjuchau/data_files/HST/ngc5194/F555W_HST_ACS_WFC_IVM_drc.fits')
ngc1672.load_image('f658', '/project/galaxies/tjuchau/data_files/HST/ngc5194/ngc1672_acs_f658n_ivm_drc_sci.fits')
ngc1672.save_fits('f555', '/project/galaxies/tjuchau/data_files/HST/ngc5194/F555W_HST_ACS_WFC_IVM_drc.fits', scale = 1.275)
ngc1672.load_image('f555_scaled', '/project/galaxies/tjuchau/data_files/HST/ngc5194/F555W_HST_ACS_WFC_IVM_drc.fits')
ngc1672.load_image('pa_cont', '/project/galaxies/tjuchau/data_files/JWST/PHANGS_products/hlsp_4793_jwst_nircam_ngc1672_f187n_v4p1_line-atf150wxf187nxf200w.fits', hdu='CONTINUUM')
jwst_cont = '/project/galaxies/tjuchau/data_files/JWST/images/ngc1672/ngc1672_nircam_lv3_f150w_i2d_anchor.fits'
ngc1672.align_images('f187', 'pa_cont')
ngc1672.align_images('f658', 'f555')
print(f'Loaded galaxy ngc1672 with callable keys :\n{list(ngc1672.images.keys())}')


In [ ]:
file = '/project/galaxies/tjuchau/data_files/JWST/PHANGS_products/hlsp_4793_jwst_nircam_ngc1512_f187n_v4p1_line-atf150wxf187nxf200w.fits'
fits.open(file).info()

In [ ]:
import obszugang
from obszugang import cluster_cat_access
x = cluster_cat_access.ClusterCatAccess()
ra,dec = x.get_hst_cc_coords_world(target = 'ngc1672')
i = 110
show_images([ngc1672.files['f150_convolved'], ngc1672.files['f187'], ngc1672.files['f555']], [ra[i],dec[i]], 0.75*u.arcsec, ncols=2)

In [ ]:
# Interactive Paschen-alpha inspection
pcs.inspect_continuum_subtraction(
    feature_name='feature',
    continuum_name='fft_conv',
    initial_scale=1.072,
    zoom_size=500,
    mask_x=5200,
    mask_y=5200,
    mask_radius=250,
    #show_all=True
)

In [ ]:
# Interactive Paschen-alpha inspection without correction
pcs.inspect_continuum_subtraction(
    feature_name='feature',
    continuum_name='cont_aligned',
    initial_scale=1.072,
    zoom_size=500,
    mask_x=5200,
    mask_y=5200,
    mask_radius=250,
    #show_all=True
)

In [ ]:
# Interactive inspection
hcs.inspect_continuum_subtraction(
    feature_name='feature',
    continuum_name='cont_alligned',
    initial_scale=0.251,
    zoom_size=None,
    mask_x=None,
    mask_y=None,
    mask_radius=None
)

In [ ]:
pcs.continuum_subtract('feature', 'fft_conv', 1.072, output_name='cont_subtracted')
pcs.save_fits('cont_subtracted', '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_continuum_subtracted.fits')
pcs.save_fits('fft_conv', '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_continuum.fits')

#hcs.continuum_subtract('feature', 'cont_alligned', 0.251, output_name='cont_subtracted')
#hcs.save_fits('cont_subtracted', '/project/galaxies/tjuchau/data_files/HST/ngc5194/F658N_HST_continuum_subtracted.fits')


In [ ]:
Ha = fits.open('/project/galaxies/tjuchau/data_files/HST/ngc5194/F658N_HST_continuum_subtracted.fits')[0].data
Pa = fits.open('/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_continuum_subtracted.fits')[0].data
ha_v = np.nanpercentile(np.abs(Ha), 99)
pa_v = np.nanpercentile(np.abs(Pa), 99)
fig, axes = plt.subplots(1,2, figsize=(8, 4))

im0 = axes[0].imshow(
    Ha,
    origin='lower',
    cmap='RdBu_r',
    vmin=-ha_v,
    vmax=ha_v
)
plt.colorbar(
            im0,
            ax=axes[0],
            fraction=0.046
        )
axes[0].axis('off')

im1 = axes[1].imshow(
    Pa,
    origin='lower',
    cmap='RdBu_r',
    vmin=-pa_v,
    vmax=pa_v
)
plt.colorbar(
            im1,
            ax=axes[1],
            fraction=0.046
        )
axes[1].axis('off')


axes[0].set_title('Ha')
axes[1].set_title('Pa')
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
import os
gal_names = ["ngc1433", 'ngc1512', 'ngc1672', "M51"]
galaxy_name = gal_names[3]


def continuum_subtract_f187(
    f150_file,
    f187_file,
    scale_factor,
    output_file=None
):
    """
    Continuum subtract a JWST F187N image using a scaled F150W image.

    Parameters
    ----------
    f150_file : str
        Path to the F150W FITS image.

    f187_file : str
        Path to the F187N FITS image.

    scale_factor : float
        Multiplicative scale factor applied to the F150W image.

    output_file : str, optional
        Output FITS filename.
        If None, a default filename is generated.

    Returns
    -------
    subtracted : ndarray
        Continuum-subtracted image array.
    """

    # ========================================================
    # LOAD DATA
    # ========================================================

    with fits.open(f150_file) as hdul150:
        f150_data = hdul150['SCI'].data.astype(float)
        f150_header = hdul150['SCI'].header

    with fits.open(f187_file) as hdul187:
        f187_data = hdul187['SCI'].data.astype(float)
        f187_header = hdul187['SCI'].header

    # ========================================================
    # CHECK SHAPES
    # ========================================================

    if f150_data.shape != f187_data.shape:
        raise ValueError(
            f'Image shapes do not match: '
            f'F150W {f150_data.shape} vs '
            f'F187N {f187_data.shape}'
        )

    # ========================================================
    # CONTINUUM SUBTRACTION
    # ========================================================

    continuum = scale_factor * f150_data

    subtracted = f187_data - continuum

    # ========================================================
    # OUTPUT FILENAME
    # ========================================================

    if output_file is None:

        base = os.path.splitext(os.path.basename(f187_file))[0]

        output_file = (
            f'{base}_contsub_scale{scale_factor:.5f}.fits'
        )

    # ========================================================
    # CREATE OUTPUT HEADER
    # ========================================================

    out_header = f187_header.copy()

    out_header['HISTORY'] = 'Continuum subtraction performed'
    out_header['CONTFILE'] = os.path.basename(f150_file)
    out_header['CONTSCL'] = scale_factor
    out_header['BUNIT'] = 'Continuum-subtracted flux'

    # ========================================================
    # WRITE FITS FILE
    # ========================================================

    hdu = fits.PrimaryHDU(
        data=subtracted,
        header=out_header
    )

    hdu.writeto(output_file, overwrite=True)

    print(f'Saved continuum-subtracted image:')
    print(f'    {output_file}')

    return subtracted

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from astropy.io import fits
from matplotlib.widgets import Slider

# ============================================================
# USER INPUTS
# ============================================================
galaxy_name = "M51"
if galaxy_name == "M51":
    hst_file = '/project/galaxies/tjuchau/data_files/HST/ngc5194/F555W_NGC5194_ACS_WFC_drc.fits'
    f150_file = '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f150w_i2d_anchor.fits'
    f187_file = '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_i2d_anchor.fits'
    f300_file = '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f300m_i2d_anchor.fits'
    cont_sub_file = '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_i2d_anchor_cont_subtracted.fits'
    out_file = '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_cont_sub.fits'
elif galaxy_name == 'ngc1433':
    hst_file = '/project/galaxies/tjuchau/data_files/HST/ngc1433/hlsp_phangs-hst_hst_wfc3-uvis_ngc1433_f555w_v1_exp-drc-sci.fits'
    f150_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1433/ngc1433_nircam_lv3_f150w_i2d_anchor.fits'
    f187_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1433/ngc1433_nircam_lv3_f187n_i2d_anchor.fits'
    f300_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1433/ngc1433_nircam_lv3_f300m_i2d_anchor.fits'
    cont_sub_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1433/ngc1433_nircam_lv3_f187n_i2d_anchor_cont_subtracted.fits'
    out_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1433/ngc1433_nircam_lv3_f187n_cont_sub.fits'

elif galaxy_name == 'ngc1512':
    hst_file = '/project/galaxies/tjuchau/data_files/HST/ngc1512/hlsp_phangs-hst_hst_wfc3-uvis_ngc1512mosaic_f555w_v1_exp-drc-sci.fits'
    f150_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1512/ngc1512_nircam_lv3_f150w_i2d_anchor.fits'
    f187_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1512/ngc1512_nircam_lv3_f187n_i2d_anchor.fits'
    f300_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1512/ngc1512_nircam_lv3_f300m_i2d_anchor.fits'
    cont_sub_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1512/ngc1512_nircam_lv3_f187n_i2d_anchor_cont_subtracted.fits'
    out_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1512/ngc1512_nircam_lv3_f187n_cont_sub.fits'

elif galaxy_name == 'ngc1672':
    hst_file = '/project/galaxies/tjuchau/data_files/HST/ngc1672/hlsp_phangs-hst_hst_wfc3-uvis_ngc1672mosaic_f555w_v1_exp-drc-sci.fits'
    f150_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1672/ngc1672_nircam_lv3_f150w_i2d_anchor.fits'
    f187_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1672/ngc1672_nircam_lv3_f187n_i2d_anchor.fits'
    f300_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1672/ngc1672_nircam_lv3_f300m_i2d_anchor.fits'
    cont_sub_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1672/ngc1672_nircam_lv3_f187n_i2d_anchor_cont_subtracted.fits'
    out_file = '/project/galaxies/tjuchau/data_files/JWST/images/NGC1672/ngc1672_nircam_lv3_f187n_cont_sub.fits'

#f150_file = '/project/galaxies/tjuchau/data_files/HST/ngc5194/F689M_HST_WFC3_UVIS_IVM_drc.fits'
#f187_file = '/project/galaxies/tjuchau/data_files/HST/ngc5194/F658N_HST_ACS_WFC_IVM_drc.fits'

try:
    f150 = fits.open(f150_file)['SCI'].data.astype(float)
    f187 = fits.open(f187_file)['SCI'].data.astype(float)
    f300 = fits.open(f300_file)['SCI'].data.astype(float)
except:
    f150 = fits.open(f150_file)[0].data.astype(float)
    f187 = fits.open(f187_file)[0].data.astype(float)
    f300 = fits.open(f300_file)[0].data.astype(float)
# Initial scale factor
initial_scale = 1.044

# Zoom region

# Set these manually after inspecting image size
x1, x2 = int(f150.shape[0]//2)-300, int(f150.shape[0]//2)+800
y1, y2 = int(f150.shape[1]//2)-300, int(f150.shape[1]//2)+800
mask_x = 5200
mask_y = 5200
mask_radius = 250

# ============================================================
# CREATE AGN MASK
# ============================================================

yy, xx = np.indices(f187.shape)

r = np.sqrt((xx - mask_x)**2 + (yy - mask_y)**2)

mask = r <= mask_radius

# Mask values with NaN
f187[mask] = np.nan
f150[mask] = np.nan

# ============================================================
# CUTOUT REGION
# ============================================================

f187_cut = f187[y1:y2, x1:x2]
f150_cut = f150[y1:y2, x1:x2]

# ============================================================
# INITIAL CONTINUUM MODEL
# ============================================================

continuum = initial_scale * f150_cut
subtracted = f187_cut - continuum

# ============================================================
# NORMALIZATION
# ============================================================

combined = np.concatenate([
    f187_cut[np.isfinite(f187_cut)].ravel(),
    continuum[np.isfinite(continuum)].ravel(),
    f150_cut[np.isfinite(f150_cut)].ravel()
])

vmin = np.percentile(combined, 1)
vmax = np.percentile(combined, 99.7)

sub_v = np.nanpercentile(np.abs(subtracted), 99)

# ============================================================
# FIGURE
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(8, 8))
axes = axes.ravel()
plt.subplots_adjust(bottom=0.15)

# ------------------------------------------------------------
# F187N
# ------------------------------------------------------------

im0 = axes[0].imshow(
    f187_cut,
    origin='lower',
    cmap='gray',
    vmin=vmin,
    vmax=vmax
)

axes[0].set_title('F187N')
axes[0].axis('off')

# ------------------------------------------------------------
# Continuum
# ------------------------------------------------------------

im1 = axes[1].imshow(
    continuum,
    origin='lower',
    cmap='gray',
    vmin=vmin,
    vmax=vmax
)

title1 = axes[1].set_title(f'Continuum = {initial_scale:.5f} × F150W')
axes[1].axis('off')

# ------------------------------------------------------------
# Subtracted
# ------------------------------------------------------------

im2 = axes[2].imshow(
    subtracted,
    origin='lower',
    cmap='RdBu_r',
    vmin=-sub_v,
    vmax=sub_v
)

title2 = axes[2].set_title('F187N - Continuum')
axes[2].axis('off')

# ------------------------------------------------------------
# f150
# ------------------------------------------------------------

im3 = axes[3].imshow(
    f150_cut,
    origin='lower',
    cmap='gray',
    vmin=-vmin,
    vmax=vmax
)

axes[3].set_title('F150W')
axes[3].axis('off')

# ============================================================
# COLORBARS
# ============================================================

plt.colorbar(im0, ax=axes[0], fraction=0.046)
plt.colorbar(im1, ax=axes[1], fraction=0.046)
plt.colorbar(im2, ax=axes[2], fraction=0.046)
plt.colorbar(im3, ax=axes[3], fraction=0.046)

# ============================================================
# SLIDER
# ============================================================

ax_slider = plt.axes([0.2, 0.05, 0.6, 0.03])

scale_slider = Slider(
    ax=ax_slider,
    label='Scale Factor',
    valmin=0.01,
    valmax=2,
    valinit=initial_scale,
    valstep=0.001
)

# ============================================================
# UPDATE FUNCTION
# ============================================================

def update(val):

    scale = scale_slider.val

    continuum_new = scale * f150_cut
    subtracted_new = f187_cut - continuum_new

    im1.set_data(continuum_new)
    im2.set_data(subtracted_new)

    sub_v_new = np.nanpercentile(np.abs(subtracted_new), 99)

    im2.set_clim(-sub_v_new, sub_v_new)

    title1.set_text(f'Continuum = {scale:.5f} × F150W')
    title2 = axes[2].set_title(f'F187N - Continuum <{np.nanmedian(subtracted_new)}>')

    fig.canvas.draw_idle()

scale_slider.on_changed(update)

plt.show()

In [ ]:
from astropy.table import Table
cont_file = '/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_continuum.fits'
table = Table.read('/project/galaxies/tjuchau/data_files/Kiana_Cluster_Files/full_table_ryan.csv')
table = table[table['galaxy'] == "M51"]
ages = []
ews = []
for row in table:
    loc = [row['ra'], row['dec']]
    radius = row['radius']*u.arcsec
    old_ew, f187_fnu, f150_fnu, f300_fnu = get_EW_using_filters(f187_file, [f150_file, f300_file], loc, radius)
    new_ew, *_ = get_equivalent_width(f187_file, cont_file, loc, radius)
    ages.append(row['best.sfh.age'])
    ews.append(new_ew)
    print(old_ew, new_ew)


In [ ]:
fig, ax = plt.subplots()
ax.scatter(ages, [i.value for i in ews])
ax.set_xscale('log')
fig.show()

In [ ]:
ews

In [ ]:
fits.open('/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f150w_i2d_anchor.fits')['SCI'].header

In [ ]:
print(fits.open('/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f187n_i2d_anchor.fits')['SCI'].header['CRVAL1'])
print(fits.open('/project/galaxies/tjuchau/data_files/JWST/images/v0p3p2/ngc5194/ngc5194_nircam_lv3_f150w_i2d_anchor.fits')['SCI'].header['CRVAL1'])